In [64]:
# Importe le module OS pour interagir avec le système d'exploitation
import os
 # Change le répertoire de travail vers le dossier parent
os.chdir("../data")

%pwd


'C:\\Cours\\Programming\\AI\\Gen_AI\\End-to-End-Medical-chatbot-Generative-AI\\data'

In [65]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader  
from langchain.text_splitter import RecursiveCharacterTextSplitter  

In [66]:
# Extract data from PDF files
def load_pdf_file(data):
    loader = DirectoryLoader(data,  
                             glob="*.pdf",
                             loader_cls=PyPDFLoader)

    documents = loader.load()

    return documents

In [67]:
extracted_data = load_pdf_file(data="../data/")

In [68]:
# extracted_data

In [69]:
#Split the Data into chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [70]:
text_chunks = text_split(extracted_data)
print("Length of text Chunks", len(text_chunks))

Length of text Chunks 5860


In [ ]:
!pip install -U langchain-huggingface
!pip uninstall sentence-transformers huggingface-hub langchain -y
!pip install sentence-transformers
!pip install huggingface-hub
!pip install langchain

In [72]:
from langchain_huggingface import HuggingFaceEmbeddings

In [73]:

# Download the embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings


In [74]:
embeddings = download_hugging_face_embeddings()

In [ ]:
# Génère l'embedding (vecteur numérique) pour la requête "Hello world" à l'aide du modèle d'embedding chargé.  
query_result = embeddings.embed_query("Hello world")  

# Affiche la longueur du vecteur généré, qui correspond au nombre de dimensions dans l'espace d'embedding.  
print("Length", len(query_result))  

import json
print(json.dumps(query_result, indent=4))

In [85]:
from dotenv import load_dotenv
load_dotenv()

True

In [86]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
OPENAI_API_KEY=os.environ.get('OPENAI_API_KEY')

In [ ]:

from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

# Initialisation de la connexion à Pinecone avec la clé API stockée dans les variables d'environnement.  
pc = Pinecone(
    api_key=PINECONE_API_KEY,
    environment="us-east-1",  # Remplacez par votre région
    timeout=180  # Spécifiez ici le délai d'attente
)  

# Nom de l'index à créer dans la base de données vectorielle Pinecone.  
index_name = "medicalbot"  

# Création d'un nouvel index dans Pinecone avec des paramètres spécifiques.  
pc.create_index(
    name=index_name,  # Nom de l'index utilisé pour identifier et stocker les vecteurs.  
    dimension=384,  # Dimension des vecteurs (à remplacer par la dimension réelle du modèle, par exemple 384).  
    metric="cosine",  # Métrique utilisée pour mesurer la similarité entre vecteurs (cosine dans ce cas).  
    spec=ServerlessSpec(  # Configuration du déploiement serverless (sans gestion explicite du serveur).  
        cloud="aws",  # Fournisseur cloud où l'index est hébergé (ici Amazon Web Services).  
        region="us-east-1"  # Région géographique pour optimiser la latence et la disponibilité.  
    ) 
)


In [ ]:
!pip install --upgrade langchain
!pip install --upgrade langchain-pinecone
!pip install --upgrade pinecone-client
!pip install --upgrade urllib3


In [ ]:

# Intégrer chaque segment et insérer les embeddings dans votre index Pinecone.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

In [ ]:
# Charger un index existant dans Pinecone

# Importation de la classe PineconeVectorStore depuis langchain_pinecone
from langchain_pinecone import PineconeVectorStore

# Intégrer chaque segment et insérer les embeddings dans l'index Pinecone
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,  # Nom de l'index existant dans Pinecone
    embedding=embeddings,   # Modèle d'embedding utilisé pour la correspondance sémantique
)
docsearch

In [ ]:
# Créer un récupérateur de documents basé sur la similarité
retriever = docsearch.as_retriever(
    search_type="similarity",       # Utiliser la recherche basée sur la similarité sémantique
    search_kwargs={"k": 3}         # Renvoyer les 3 documents les plus similaires
)

# Effectuer une recherche avec une requête textuelle
retrieved_docs = retriever.invoke("What is Acne?")  # Recherche des documents liés à l'acné


In [ ]:
retrieved_docs

In [87]:
# Importation du modèle OpenAI depuis langchain_openai
from langchain_openai import OpenAI

# Initialisation du modèle OpenAI avec des paramètres personnalisés
llm = OpenAI(
    temperature=0.4,   # Contrôle la créativité des réponses (0.4 = légèrement créatif mais cohérent)
    max_tokens=500     # Limite la longueur maximale de la réponse à 500 tokens
)